# Titanic Survival Prediction – Complete Project

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## 🧭 Day 1: Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
train = pd.read_csv('/kaggle/input/titanic/train.csv')
test = pd.read_csv('/kaggle/input/titanic/test.csv')

## 1. Data Overview

In [ ]:
# print(train.shape)
# print(train.head())
print(train.info())
print(test.info())
# print(train.describe())

## 2. Missing Values

In [ ]:
print(train.isnull().sum())
print(test.isnull().sum())

## 3. Missing-Value Visualization

In [ ]:
plt.figure(figsize=(8,5))
sns.heatmap(train.isnull(), cbar=False, cmap='mako')
plt.title('Missing Values Heatmap')
plt.show()

plt.figure(figsize=(8,5))
sns.heatmap(test.isnull(), cbar=False, cmap='mako')
plt.title('Missing Values Heatmap')
plt.show()

**Observation:**<br>
The **Cabin** column has the highest number of missing values (~77%), while **Age** has around 19% missing.<br>
This will need imputation or feature engineering later.

## 4. Exploratory Data Analysis (EDA)

## Visualizations

### Survival distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='Survived', data=train)
plt.title('Survival Count')
plt.show()

pd.DataFrame(train['Survived'].value_counts(normalize=True).round(2) * 100)

**Observation:** <br>
Around **38% of passengers survived**, while **62% did not**.<br>
The dataset is slightly imbalanced, with more non-survivors, indicating rescue chances were relatively low overall.

### Gender vs Survival

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='Sex', hue='Survived', data=train)
plt.title('Survival by Gender')
plt.show()

pd.DataFrame(train.groupby('Sex')['Survived'].mean().round(2) * 100)

**Observation:** <br>
Females have a 74% survival rate vs only 19% for males.  
This confirms that gender was a strong predictor of survival,aligning with the “women and children first” evacuation rule.

### Class vs Survival

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='Pclass', hue='Survived', data=train)
plt.title('Survival by Passenger Class')
plt.show()

pd.DataFrame(train.groupby('Pclass')['Survived'].mean().round(2) * 100)

**Observation:**<br>
1st-class passengers had the highest survival rate (63%), while 3rd-class had the lowest (~24%).<br>
This indicates that wealth and cabin location influenced rescue chances.

### Age vs Survival

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(x='Age', hue='Survived', data=train, kde=True)
plt.title('Survival by Age')
plt.show()

pd.DataFrame(train.groupby(pd.cut(train['Age'], bins=[0,12,18,30,50,80]))['Survived'].mean().round(2) * 100
)

**Observation:**<br>
Children (below 12 years) had the **highest survival rates**, while middle-aged adults survived less frequently.<br>
This suggests **younger passengers were prioritized** during evacuation.

### Fare vs Survival

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='Survived', y='Fare', data=train)
plt.title('Survival by Fare')
plt.show()

pd.DataFrame(train.groupby(pd.cut(train['Fare'], bins=[0,50,100,250,600]))['Survived'].mean().round(2) * 100
)

**Observation:**<br>
Passengers who paid higher fares had **better survival chances.**<br>
This supports the idea that **wealthier passengers (with expensive cabins)** were located closer to lifeboats or assisted earlier.

### Siblings/Spouses vs Survival

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='SibSp', hue='Survived', data=train)
plt.title('Survival by Siblings/Spouses')
plt.show()

pd.DataFrame(train.groupby('SibSp')['Survived'].mean().round(2) * 100)

**Observation:**<br>
Passengers with **1 or 2 siblings/spouses** aboard had **higher survival rates**.<br>
Traveling completely alone or with large families (≥3) reduced chances of survival — possibly due to coordination issues or panic.

### Parent/Child vs Survival

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='Parch', hue='Survived', data=train)
plt.title('Survival by Parent/Child')
plt.show()

pd.DataFrame(train.groupby('Parch')['Survived'].mean().round(2) * 100)

**Observation:**<br>
Having **1–2 parents or children** increased survival probability, while being alone or in large family groups lowered it.<br>
This again shows that **small families** had better survival odds, perhaps due to mutual support during evacuation.

## 5. Summary of Insights


- Women and children had much higher survival chances.
- 1st-class passengers survived more often than lower classes.
- Higher fares and smaller families increased survival probability.
- Missing data in Age and Cabin will need careful handling.

## ⚙️ Day 2: Data Cleaning & Feature Engineering

## 🪜 Step 1 – Handle Missing Values

In [ ]:
# Fill Age with median grouped by Sex and Pclass
train['Age'] = train.groupby(['Sex', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))
test['Age'] = test.groupby(['Sex', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))

# Fill Embarked with mode
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])

# Fill missing Fare (test set)
test['Fare'] = test['Fare'].fillna(test['Fare'].median())

**Insight:** <br>

* Missing **Age** values were imputed using median based on **Sex** and **Pclass** to maintain demographic patterns.
* Missing **Embarked** filled with mode (“S”), the most common port.
* **Fare** missing value filled with median to preserve distribution.

## 🧩 Step 2 – Feature Engineering

### ✨ 2.1 Family Features

In [ ]:
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1

train['IsAlone'] = (train['FamilySize'] == 1).astype(int)
test['IsAlone'] = (test['FamilySize'] == 1).astype(int)

**🧠 Why: Family context affects survival — small families coordinated better.**

### ✨ 2.2 Title Extraction (Optional Bonus Feature)

In [ ]:
train['Title'] = train['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
test['Title'] = test['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

title_map = {'Mr':0, 'Miss':1, 'Mrs':2, 'Master':3}
train['Title'] = train['Title'].map(title_map)
test['Title'] = test['Title'].map(title_map)
train['Title'] = train['Title'].fillna(0)
test['Title'] = test['Title'].fillna(0)

**🧠 Why: The title encodes social status and gender (useful for survival).**

### ✨ 2.3 Encoding Categorical Variables

In [ ]:
# Encode Sex
train['Sex'] = train['Sex'].map({'male':0, 'female':1})
test['Sex'] = test['Sex'].map({'male':0, 'female':1})

# One-hot encode Embarked
train = pd.get_dummies(train, columns=['Embarked'], drop_first=True)
test = pd.get_dummies(test, columns=['Embarked'], drop_first=True)

**🧠 Why: ML models can’t interpret text — we convert categories to numbers.**

### ✨ 2.4 Drop Unnecessary Columns

In [ ]:
cols_to_drop = ['Name', 'Ticket', 'Cabin', 'PassengerId']
train.drop(columns=cols_to_drop, inplace=True)
test.drop(columns=cols_to_drop, inplace=True)

**🧠 Why: These columns add noise or are identifiers.**

## ⚙️ Step 3 – Model Preparation

In [ ]:
X = train.drop('Survived', axis=1)
y = train['Survived']

In [ ]:
# Split data
from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

## 🤖 Step 4 – Train Your First Model (Logistic Regression)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_valid)

print("Accuracy:", accuracy_score(y_valid, y_pred))
print(confusion_matrix(y_valid, y_pred))
print(classification_report(y_valid, y_pred))


**🧠 Why Logistic Regression?<br>It’s the simplest yet effective baseline for classification — interpretable and quick.**


## 📊 Step 5 – Generate Kaggle Submission

In [ ]:
# Re-add PassengerId (dropped it earlier, so load again)
test_full = pd.read_csv('/kaggle/input/titanic/test.csv')
X_test = test
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test_full['PassengerId'],
    'Survived': predictions
})

submission.to_csv('submission.csv', index=False)

# Summary
* Missing data handled logically using grouped medians and modes.
* Created new features (FamilySize, IsAlone, Title) that improved interpretability.
* Encoded categorical data properly.
* Trained a baseline Logistic Regression achieving ~77–80% accuracy (depending on seed).
* Next step: experiment with Decision Trees, Random Forest, and feature scaling.